# Evologics USBL — Position Visualisation

Plots ship track and resolved target positions on an interactive map, and
shows time series of target latitude/longitude, local XYZ offsets, and depth
for a single deployment.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "qd61g27j_20170523_040815"

# NOTE: Deployments with Evologics USBL
# qd61g27j_20170523_040815
# qdc5ghs3_20210315_230947
# qdch0ftq_20170526_025746
# qdch0ftq_20210315_034028
# qdchdmy1_20170525_234624
# qdchdmy1_20210315_081519
# qtqxshxs_20150327_015552
# qtqxshxs_20150328_000850
# qtqxshxs_20150328_042551

USBL_DIR: Path = Path("/data/exos_01/acfr_messages_v2_parsed")
USBL_FILE: Path = USBL_DIR / f"{DEPLOYMENT}_usbl_evologics.csv"

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "target_latitude",
    "target_longitude",
    "target_depth",
    "target_x",
    "target_y",
    "target_z",
    "accuracy",
    "ship_latitude",
    "ship_longitude",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)

missing = [col for col in REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

print(f"Rows: {len(usbl)}")
usbl.head(3)

## Overview map: ship track and target positions

In [ ]:
t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl["target_latitude"].mean())
center_lon: float = float(usbl["target_longitude"].mean())

colorscale: str = "Viridis"

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["target_latitude"],
        lon=usbl["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [usbl["target_depth"], usbl["accuracy"]],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Accuracy: %{customdata[1]:.2f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Ship track and USBL target positions — {DEPLOYMENT}",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Time series: target latitude and longitude

In [ ]:
fig_latlon = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("Target latitude (°)", "Target longitude (°)"),
    vertical_spacing=0.08,
)

fig_latlon.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_latitude"],
        mode="lines+markers",
        marker=dict(size=4),
        name="Target latitude",
        line=dict(color="steelblue"),
    ),
    row=1,
    col=1,
)

fig_latlon.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["ship_latitude"],
        mode="lines+markers",
        marker=dict(size=4),
        name="Ship latitude",
        line=dict(color="steelblue", dash="dash"),
    ),
    row=1,
    col=1,
)

fig_latlon.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_longitude"],
        mode="lines+markers",
        marker=dict(size=4),
        name="Target longitude",
        line=dict(color="darkorange"),
    ),
    row=2,
    col=1,
)

fig_latlon.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(size=4),
        name="Ship longitude",
        line=dict(color="darkorange", dash="dash"),
    ),
    row=2,
    col=1,
)

fig_latlon.update_layout(
    title=f"Target and ship latitude/longitude — {DEPLOYMENT}",
    height=550,
    legend=dict(x=0.01, y=0.99),
)

fig_latlon.show()

## Time series: target X, Y, Z offsets and depth

In [ ]:
fig_xyz = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Target X offset (m)",
        "Target Y offset (m)",
        "Target Z offset (m)",
        "Target depth (m)",
    ),
    vertical_spacing=0.06,
)

for row, (col, color) in enumerate(
    [
        ("target_x", "steelblue"),
        ("target_y", "darkorange"),
        ("target_z", "seagreen"),
        ("target_depth", "crimson"),
    ],
    start=1,
):
    fig_xyz.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=usbl[col],
            mode="lines+markers",
            marker=dict(size=4),
            name=col,
            line=dict(color=color),
        ),
        row=row,
        col=1,
    )

fig_xyz.update_yaxes(autorange="reversed", row=4, col=1)

fig_xyz.update_layout(
    title=f"Target XYZ offsets and depth — {DEPLOYMENT}",
    height=900,
    showlegend=False,
)

fig_xyz.show()